## Vector DB + Embeddings Demo (Pinecone + pgvector)

This notebook is split into 3 parts:
- **Embeddings (local)**: Sentence-BERT `all-MiniLM-L6-v2`
- **Pinecone**: create index → upsert vectors → query top-k → metadata filtering
- **pgvector (PostgreSQL)**: vector column → upsert → distance operators → top-k + filters

Prereqs:
- You already have a Python `venv` at `../venv/`
- Pinecone section requires `PINECONE_API_KEY`
- pgvector section requires a Postgres connection string `PG_DSN` to a DB with `pgvector` extension installed (or permissions to install it).


In [1]:
# If you get import errors, run this cell once.
# (Safe to re-run; pip will no-op if already installed.)

%pip -q install "pinecone>=5" "openai>=1.0" "great-expectations>=0.18" "pgvector>=0.3"  # pgvector python helpers


Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
from typing import List, Dict, Any

import numpy as np
from sentence_transformers import SentenceTransformer

# Sample documents
DOCS: List[Dict[str, Any]] = [
    {
        "id": "doc-1",
        "text": "Refunds are processed within 5 business days.",
        "metadata": {"topic": "policy", "lang": "en", "source": "help-center"},
    },
    {
        "id": "doc-2",
        "text": "Shipping usually takes 2 to 4 days in metro cities.",
        "metadata": {"topic": "shipping", "lang": "en", "source": "help-center"},
    },
    {
        "id": "doc-3",
        "text": "For damaged items, submit photos within 48 hours of delivery.",
        "metadata": {"topic": "returns", "lang": "en", "source": "policy"},
    },
    {
        "id": "doc-4",
        "text": "Cash on delivery is available for orders above ₹499.",
        "metadata": {"topic": "payments", "lang": "en", "source": "checkout"},
    },
]

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = [d["text"] for d in DOCS]
emb = model.encode(texts, normalize_embeddings=True)

DIM = int(emb.shape[1])
print("Embedding dim:", DIM)

vectors = [
    {
        "id": d["id"],
        "values": emb[i].astype(np.float32).tolist(),
        "metadata": {**d["metadata"], "text": d["text"]},
    }
    for i, d in enumerate(DOCS)
]

print("Example vector payload:")
print(json.dumps({k: vectors[0][k] for k in ("id", "metadata")}, indent=2))
print("values[0:5] =", vectors[0]["values"][:5])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dim: 384
Example vector payload:
{
  "id": "doc-1",
  "metadata": {
    "topic": "policy",
    "lang": "en",
    "source": "help-center",
    "text": "Refunds are processed within 5 business days."
  }
}
values[0:5] = [-0.0612926185131073, -0.004709804896265268, 0.10437336564064026, -0.0042523955926299095, 0.046693556010723114]


In [4]:
# Quick local semantic search (no DB): cosine similarity top-k

def topk_local(query: str, k: int = 3, topic: str | None = None):
    qv = model.encode([query], normalize_embeddings=True)[0]
    scores = []
    for i, d in enumerate(DOCS):
        if topic and d["metadata"].get("topic") != topic:
            continue
        s = float(np.dot(qv, emb[i]))  # cosine since normalized
        scores.append((s, d))
    scores.sort(reverse=True, key=lambda x: x[0])
    return scores[:k]

for score, d in topk_local("How long do refunds take?", k=3):
    print(round(score, 4), d["id"], d["text"])

0.9 doc-1 Refunds are processed within 5 business days.
0.2731 doc-3 For damaged items, submit photos within 48 hours of delivery.
0.2534 doc-2 Shipping usually takes 2 to 4 days in metro cities.


## Pinecone (Index → Upsert → Query → Metadata filter)

Set environment variable `PINECONE_API_KEY`.

Notes:
- Index **dimension must match** the embedding dimension printed above.
- We use cosine similarity by storing **normalized** embeddings (works well with dot-product/cosine).

In [15]:
import os
from dotenv import load_dotenv

load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")
pinecone_key = os.getenv("PINECONE_API_KEY")

print("Keys loaded ✅")

Keys loaded ✅


In [3]:
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
import uuid
import os

# Load API key
from dotenv import load_dotenv
load_dotenv()

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

model = SentenceTransformer('all-MiniLM-L6-v2')

index_name = "product-search"

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,   # MiniLM embedding size
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("Index ready ✅")

Index ready ✅


In [ ]:
index = pc.Index(index_name)

In [25]:
products = [
    {"id": "1", "text": "Smartphone with long battery life", "category": "electronics"},
    {"id": "2", "text": "High performance gaming laptop", "category": "electronics"},
    {"id": "3", "text": "Comfortable cotton t-shirt", "category": "clothing"},
    {"id": "4", "text": "Phone with excellent camera quality", "category": "electronics"},
    {"id": "5", "text": "Affordable budget smartphone", "category": "electronics"},
]

In [26]:
texts = [p["text"] for p in products]
embeddings = model.encode(texts)

In [27]:
vectors = []

for i, p in enumerate(products):
    vectors.append({
        "id": p["id"],
        "values": embeddings[i].tolist(),
        "metadata": {
            "text": p["text"],
            "category": p["category"]
        }
    })

index.upsert(vectors=vectors)

print("Data inserted ✅")

Data inserted ✅


In [ ]:
query = "good battery phone"

query_emb = model.encode([query])[0]

results = index.query(
    vector=query_emb.tolist(),
    top_k=3,
    include_metadata=True
)

print(results)

QueryResponse(matches=[{'id': '4',
 'metadata': {'category': 'electronics',
              'text': 'Phone with excellent camera quality'},
 'score': 0.658072531,
 'values': []}, {'id': '1',
 'metadata': {'category': 'electronics',
              'text': 'Smartphone with long battery life'},
 'score': 0.614439,
 'values': []}, {'id': '5',
 'metadata': {'category': 'electronics',
              'text': 'Affordable budget smartphone'},
 'score': 0.531487942,
 'values': []}], namespace='', usage={'read_units': 1}, _response_info={'raw_headers': {'date': 'Tue, 07 Apr 2026 12:49:34 GMT', 'content-type': 'application/json', 'content-length': '431', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '2', 'x-pinecone-request-latency-ms': '41', 'x-envoy-upstream-service-time': '42', 'x-pinecone-response-duration-ms': '43', 'grpc-status': '0', 'server': 'envoy'}})


In [31]:
for match in results["matches"]:
    print(f"Score: {match['score']:.3f}")
    print(f"Text: {match['metadata']['text']}")
    print("------")

Score: 0.658
Text: Phone with excellent camera quality
------
Score: 0.614
Text: Smartphone with long battery life
------
Score: 0.531
Text: Affordable budget smartphone
------


In [20]:
results = index.query(
    vector=query_emb.tolist(),
    top_k=3,
    include_metadata=True,
    filter={"category": "electronics"}
)

for match in results["matches"]:
    print(match["metadata"]["text"])

NameError: name 'index' is not defined

In [33]:
!pip install openai psycopg2-binary sqlalchemy

In [4]:
import psycopg2
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [18]:
conn = psycopg2.connect(
    dbname="postgres",
    user="navalyemul",   # ✅ FIX
    password="",         # usually empty on Mac
    host="localhost",
    port="5432"
)

cursor = conn.cursor()

In [45]:
conn.rollback()

In [6]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    id SERIAL PRIMARY KEY,
    text TEXT,
    embedding VECTOR(1536)
);
""")

conn.commit()
print("Table ready ✅")

Table ready ✅


In [7]:
products = [
    "Smartphone with long battery life",
    "Gaming laptop with high performance",
    "Affordable budget phone",
    "Phone with excellent camera quality",
    "Cheap mobile with decent battery"
]

for text in products:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    
    emb = response.data[0].embedding
    
    cursor.execute(
        "INSERT INTO products (text, embedding) VALUES (%s, %s)",
        (text, emb)
    )

conn.commit()
print("Data inserted ✅")

Data inserted ✅


In [19]:
conn.rollback()

query = "good battery phone"

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
)

query_emb = response.data[0].embedding

In [10]:
query_emb

[0.0233001708984375,
 -0.0301666259765625,
 -0.0074615478515625,
 0.0288848876953125,
 -0.005706787109375,
 -0.0204620361328125,
 -0.009490966796875,
 0.0230560302734375,
 0.03125,
 -0.039642333984375,
 -0.021026611328125,
 -0.0241546630859375,
 -0.037933349609375,
 -0.015289306640625,
 0.0374755859375,
 0.0031566619873046875,
 -0.03814697265625,
 -0.022735595703125,
 0.0167236328125,
 0.0006833076477050781,
 0.013824462890625,
 0.039520263671875,
 0.060821533203125,
 -0.03411865234375,
 -0.0270233154296875,
 -0.03045654296875,
 -0.03582763671875,
 -0.02252197265625,
 0.00029778480529785156,
 -0.026702880859375,
 0.0086517333984375,
 -0.0253753662109375,
 0.0287628173828125,
 -0.04571533203125,
 -0.044769287109375,
 -0.00473785400390625,
 0.0411376953125,
 -0.04266357421875,
 0.050872802734375,
 -0.005893707275390625,
 -0.040985107421875,
 -0.0135345458984375,
 0.0151519775390625,
 0.0240936279296875,
 0.003963470458984375,
 -0.043792724609375,
 -0.0186309814453125,
 0.0243988037109375

In [49]:
query = "good battery phone"

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
)

query_emb = response.data[0].embedding

In [12]:
conn.rollback()
cursor.execute("""
SELECT id, text,
       embedding <=> %s AS distance
FROM products
ORDER BY embedding <=> %s
LIMIT 3;
""", (query_emb, query_emb))

results = cursor.fetchall()

for r in results:
    print(f"ID: {r[0]}")
    print(f"Text: {r[1]}")
    print(f"Distance: {r[2]:.4f}")
    print("------")

UndefinedFunction: operator does not exist: vector <=> numeric[]
LINE 3:        embedding <=> ARRAY[0.0233001708984375, -0.0301666259...
                         ^
HINT:  No operator matches the given name and argument types. You might need to add explicit type casts.


In [52]:
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="good battery phone"
)

embedding = response.data[0].embedding

print(len(embedding))  # 1536

1536


In [60]:
conn.rollback()

In [56]:
cursor.execute("SELECT COUNT(*) FROM products;")
print(cursor.fetchone())

(5,)


In [61]:
cursor.execute("SELECT id, text FROM products;")
print(cursor.fetchall())

[(1, 'Smartphone with long battery life'), (2, 'Gaming laptop with high performance'), (3, 'Affordable budget phone'), (4, 'Phone with excellent camera quality'), (5, 'Cheap mobile with decent battery')]


In [62]:
cursor.execute("""
SELECT id, text,
       embedding <=> %s::vector AS distance
FROM products
ORDER BY embedding <=> %s::vector
LIMIT 3;
""", (query_emb, query_emb))

In [63]:
print(type(query_emb))

<class 'list'>


In [13]:
# Step 1: Reset
conn.rollback()

# Step 2: Run query
cursor.execute("""
SELECT id, text,
       embedding <=> %s::vector AS distance
FROM products
ORDER BY embedding <=> %s::vector
LIMIT 3;
""", (query_emb, query_emb))

results = cursor.fetchall()

for r in results:
    print(f"ID: {r[0]}")
    print(f"Text: {r[1]}")
    print(f"Distance: {r[2]:.4f}")
    print("------")

ID: 5
Text: Cheap mobile with decent battery
Distance: 0.2994
------
ID: 10
Text: Cheap mobile with decent battery
Distance: 0.2994
------
ID: 1
Text: Smartphone with long battery life
Distance: 0.3305
------


In [18]:
# Upsert
index.upsert(vectors=vectors)

# Query (top-k)
query = "Is cash on delivery available?"
qv = model.encode([query], normalize_embeddings=True)[0].astype(np.float32).tolist()

res = index.query(vector=qv, top_k=3, include_metadata=True)
print("Query:", query)
for m in res.get("matches", []):
    print(round(m["score"], 4), m["id"], "topic=", m["metadata"].get("topic"), "text=", m["metadata"].get("text"))


Query: Is cash on delivery available?
0.7295 doc-4 topic= payments text= Cash on delivery is available for orders above ₹499.
0.301 doc-3 topic= returns text= For damaged items, submit photos within 48 hours of delivery.
0.2915 doc-2 topic= shipping text= Shipping usually takes 2 to 4 days in metro cities.


In [20]:
# Metadata filtering (only search within topic=shipping)
res = index.query(
    vector=qv,
    top_k=3,
    include_metadata=True,
    filter={"topic": {"$eq": "shipping"}},
)
print("\nFiltered to topic=shipping")
for m in res.get("matches", []):
    print(round(m["score"], 4), m["id"], m["metadata"].get("text"))



Filtered to topic=shipping
0.2915 doc-2 Shipping usually takes 2 to 4 days in metro cities.
